In [ ]:
from sampo.generator.base import SimpleSynthetic
from sampo.generator.environment.contractor_by_wg import get_contractor_by_wg
from sampo.scheduler.genetic.base import GeneticScheduler
from sampo.scheduler.genetic.operators import TimeAndResourcesFitness
from sampo.utilities.resource_usage import resources_peaks_sum, resources_costs_sum, resources_sum
import pandas as pd

## Set parameters and generate synthetic graph

In [ ]:
experiment_name = "adaptive_1"

graph_size = 150
seed = 123

size_of_population = 200
number_of_generation = 200

mutate_order = 0.01
mutate_resources = 0.01

fitness_constructor = TimeAndResourcesFitness()
fitness_weights = (-1, -1)
is_multiobjective = True
optimize_resources = True

save_history_to = f"history/{experiment_name}.json"
save_pareto_to = f"pareto/{experiment_name}.csv"

In [ ]:
ss = SimpleSynthetic(seed)
wg = ss.work_graph(bottom_border=graph_size)
contractors = [get_contractor_by_wg(wg)]
print(f"Generated graph with size: {wg.vertex_count}")

## Use the genetic algorithm and save history

In [ ]:
genetic_algorithm = GeneticScheduler(
    number_of_generation=number_of_generation,
    size_of_population=size_of_population,
    
    mutate_order=mutate_order,
    mutate_resources=mutate_resources,
    
    fitness_constructor=fitness_constructor,
    fitness_weights=fitness_weights,
    is_multiobjective=is_multiobjective,
    optimize_resources=optimize_resources,
    
    seed=seed,
    save_history_to=save_history_to
)
genetic_result = genetic_algorithm.schedule(wg, contractors)

pareto_front = [
    (schedule.execution_time.value, resources_peaks_sum(schedule))
    for schedule in genetic_result
]
df = pd.DataFrame(pareto_front, columns=["Time", "Cost"]).sort_values(["Time", "Cost"])
df.to_csv(save_pareto_to, index=False)

## Plot Pareto-fronts

In [ ]:
from os import listdir
import numpy as np
import pandas as pd
import json
import plotly.express as px

In [ ]:
df = pd.concat([
    pd.read_csv(f"pareto/{file}").assign(experiment=file)
    for file in listdir("pareto")
])

In [ ]:
fig = px.line(
    df, x="Time", y="Cost", color="experiment",
    template="plotly_white"
)
fig.update_traces(mode="lines+markers")

experiment_to_color = {
    "genetic1": "black",
    "genetic2": "black",
    
    "omcc1": "green",
    "omcc2": "green",
    "omcc3": "green",
    "omcc5": "green",
    "omcc4": "green",
    "om1": "blue"
}
for i, trace in enumerate(fig.data):
    for experiment, color in experiment_to_color.items():
        if trace.name.startswith(experiment):
            fig.data[i].line.color=color

fig.update_layout(height=1000, width=1000, showlegend=True)
# fig.write_image("pareto.png", scale=3)
fig.show()

In [ ]:
df.groupby("experiment").mean()

## Get summary of the evolution

In [ ]:
from sampo.scheduler.utils.fitness_history import FitnessHistorySummary

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_dark"

In [ ]:
# Load history from file
summary = FitnessHistorySummary.load_json("history/omccad_3.json")

In [ ]:
population_means = np.array(summary.get_fitness_means())

fig = px.line(y=population_means[:, 0])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
population_means = np.array(summary.get_fitness_means())

fig = px.line(y=population_means[:, 1])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
pareto_front_means = np.array(summary.get_fitness_means(only_pareto=True))

fig = px.line(y=pareto_front_means[:, 0])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
pareto_front_means = np.array(summary.get_fitness_means(only_pareto=True))

fig = px.line(y=pareto_front_means[:, 1])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
offsprings_means = np.array(summary.get_fitness_means(only_offsprings=True))

fig = px.line(y=offsprings_means[:, 0])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
offsprings_means = np.array(summary.get_fitness_means(only_offsprings=True))

fig = px.line(y=offsprings_means[:, 1])
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
pareto_ratios = summary.get_pareto_to_population_ratios()

fig = px.line(y=pareto_ratios)
fig.update_layout(height=500, width=1000)
fig.show()

In [ ]:
population_shifts = summary.get_generation_shifts()
pareto_front_shifts = summary.get_generation_shifts(only_pareto=True)

fig = px.line(y=[population_shifts, pareto_front_shifts])
fig.data[1].line.color = "white"
fig.update_layout(height=500, width=1000, showlegend=False)
fig.show()

In [ ]:
population_uniqueness = summary.get_uniqueness_scores()
pareto_uniqueness = summary.get_uniqueness_scores(only_pareto=True)

fig = px.line(y=[population_uniqueness, pareto_uniqueness])
fig.data[1].line.color = "white"
fig.update_layout(height=500, width=1000, showlegend=False)
fig.show()

In [ ]:
df = pd.DataFrame(dict(
    comments=summary.comments[1:],
    population_shifts=population_shifts
))

In [ ]:
df.groupby("comments").mean().squeeze().sort_values(ascending=False)